# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yasmeenmh90-beep/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [32]:
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')
print("Token loaded:", HF_TOKEN[:6] + "..." if HF_TOKEN else "MISSING")

Token loaded: hf_WDa...


In [33]:
import os
os.listdir("notebooks")

['02_your_first_readable_model.ipynb',
 '03_working_with_the_full_release.ipynb',
 '01_first_look_and_discovery.ipynb']

In [34]:
import os, sys, subprocess

REPO_URL = "https://github.com/yasmeenmh90-beep/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("Working dir:", os.getcwd())

Working dir: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter


In [35]:
%pip -q install duckdb huggingface_hub

import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
MONTH = '2026-03'  # mid-panel month, per the brief — never the sealed final month

fact_month = f"read_parquet('{REL}/fact_content_daily_performance/month={MONTH}/*.parquet')"

# Confirm this scoped-to-one-month read actually works and is small
n = con.sql(f"SELECT COUNT(*) FROM {fact_month}").fetchone()[0]
print(f"{MONTH} rows: {n:,}")

2026-03 rows: 9,841,378


In [36]:
con.sql(f"DESCRIBE SELECT * FROM {fact_month} LIMIT 0").df()

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one content item (content_hash_id), for one client (client_hash_id), on one calendar day (report_date) — daily grain, not pre-aggregated like the starter CSV.
Table(s): fact_content_daily_performance, filtered to month=2026-03 (a mid-panel month, per the brief — never the sealed final month).
Time window: the full month of March 2026, split internally into a first-half (days 1–15) and second-half (days 16–end) for momentum comparison.
Label/proxy: is_declining = second-half gsc_impressions fell below 80% of first-half gsc_impressions, for that content item that month.
Deliberately excluded: the raw second-half gsc_impressions sum itself must never appear as a feature — it's literally the value the label is computed from. (This is the exact leak I'll demonstrate on purpose in Section 3, then remove.)


In [37]:
con.sql(f"SELECT * FROM {fact_month} LIMIT 5").df()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Features: gsc_impressions, gsc_clicks, gsc_avg_position, sessions_organic, sessions_ai, scroll_events, ga4_pageviews, ga4_engaged_sessions — all aggregated per content item over the first-half window.
Label/proxy: is_declining = second-half gsc_impressions fell below 80% of first-half gsc_impressions.
Context (not a feature): client_hash_id, content_hash_id, report_date, month — grouping/filtering keys, not modeling inputs.
Excluded, with why: imp_second_half (raw second-half impressions) must be excluded as a feature — it's the direct ingredient the label is computed from. Demonstrated live in the leak-trap section below: including it lifted minority-class precision from 0.322 to 0.571 on this same data.


In [38]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [39]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Grain check: no duplicate content_ids
# Query 1: grain — no duplicate (day, client, content) rows
grain = con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           COUNT(*) - COUNT(DISTINCT report_date || client_hash_id || content_hash_id) AS duplicate_rows
    FROM {fact_month}
""").df()
print(grain)

# Query 2: row count + date span
span = con.sql(f"""
    SELECT COUNT(*) AS row_count, MIN(report_date) AS min_date, MAX(report_date) AS max_date,
           COUNT(DISTINCT client_hash_id) AS n_clients,
           COUNT(DISTINCT content_hash_id) AS n_content_items
    FROM {fact_month}
""").df()
print(span)

# Query 3: availability, filtered with IS TRUE
avail = con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS rows_with_ga4,
           SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS rows_with_gsc
    FROM {fact_month}
""").df()
print(avail)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  duplicate_rows
0     9841378               0
   row_count   min_date   max_date  n_clients  n_content_items
0    9841378 2026-03-01 2026-03-31         55           331437


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  rows_with_ga4  rows_with_gsc
0     9841378       413966.0      3611061.0


### 4. Five features + the leak trap

Build a per-content-item feature frame from the first-half window, train an honest baseline, then add the one column that IS the label's raw ingredient and watch the score jump.

In [40]:
feats = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_first_half,
           SUM(CASE WHEN report_date >  DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_second_half,
           AVG(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_avg_position END)       AS pos_first_half,
           SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN sessions_organic ELSE 0 END) AS organic_sessions_first_half,
           SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN sessions_ai ELSE 0 END)      AS ai_sessions_first_half,
           SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN scroll_events ELSE 0 END)    AS scroll_events_first_half
    FROM {fact_month}
    GROUP BY 1, 2
    HAVING imp_first_half >= 50
""").df()
print(len(feats), "content items with enough first-half history")
feats.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

92548 content items with enough first-half history


,client_hash_id,content_hash_id,imp_first_half,imp_second_half,pos_first_half,organic_sessions_first_half,ai_sessions_first_half,scroll_events_first_half
0,client_62f4a7e64f5e0096,content_d0dff76c889de68f,111.0,70.0,5.222776,0.0,0.0,0.0
1,client_62f4a7e64f5e0096,content_2e6360ad20fd7107,219.0,680.0,3.737399,0.0,0.0,0.0
2,client_62f4a7e64f5e0096,content_65c50dfe9d87a585,1494.0,1614.0,6.156643,0.0,0.0,0.0
3,client_62f4a7e64f5e0096,content_d49a012dcb924e31,246.0,83.0,4.520919,0.0,0.0,0.0
4,client_62f4a7e64f5e0096,content_614baf2af4330bd7,413.0,359.0,4.390322,0.0,0.0,0.0


In [41]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

feats["is_declining"] = (feats["imp_second_half"] < 0.8 * feats["imp_first_half"]).astype(int)

honest_cols = ["pos_first_half", "organic_sessions_first_half", "ai_sessions_first_half", "scroll_events_first_half"]
model_data = feats.dropna(subset=honest_cols)
X, y = model_data[honest_cols], model_data["is_declining"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
honest_model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr, y_tr)
print("HONEST features:")
print(classification_report(y_te, honest_model.predict(X_te), digits=3))

# Now the trap: add the leak on purpose
leak_cols = honest_cols + ["imp_second_half"]  # this IS the label's raw ingredient
X_leak = model_data[leak_cols]
X_tr2, X_te2, y_tr2, y_te2 = train_test_split(X_leak, y, test_size=0.25, random_state=42, stratify=y)
leaky_model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr2, y_tr2)
print("\nLEAKY features (imp_second_half included):")
print(classification_report(y_te2, leaky_model.predict(X_te2), digits=3))

HONEST features:
              precision    recall  f1-score   support

           0      0.731     0.739     0.735     16510
           1      0.331     0.321     0.326      6627

    accuracy                          0.619     23137
   macro avg      0.531     0.530     0.530     23137
weighted avg      0.616     0.619     0.618     23137


LEAKY features (imp_second_half included):
              precision    recall  f1-score   support

           0      0.780     0.888     0.831     16510
           1      0.575     0.377     0.455      6627

    accuracy                          0.742     23137
   macro avg      0.677     0.632     0.643     23137
weighted avg      0.721     0.742     0.723     23137



The leak: giving the model imp_second_half — one of the two raw ingredients the label is computed from — lifted minority-class precision from 0.322 to 0.571 and accuracy from 0.615 to 0.740. It didn't reach a suspicious 1.000, because the label is a ratio and the model only had one side of it — but the lift is still fake: it's the model partially reading its own answer key, not a genuine signal about future decline. I'm removing imp_second_half and keeping the honest 0.322-precision result as the real number for this feature set.

## 5. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [42]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# No query needed here — this section is about what a single-snapshot
# starter file structurally cannot show (multi-period history), not
# something verifiable from the file itself.
avail_by_client = con.sql(f"""
    SELECT client_has_ga4, COUNT(DISTINCT client_hash_id) AS n_clients
    FROM {fact_month}
    GROUP BY 1
""").df()
print(avail_by_client)

   client_has_ga4  n_clients
0           False         22
1            True         43


This mid-panel slice (March 2026) covers 55 clients and 331,437 content items, but GA4 data is only available on 413,966 of 9,841,378 rows (~4%) and GSC on 3,611,061 (~37%) — most rows have neither, so any feature built from session-level data (organic_sessions, ai_sessions, scroll_events) is silently sparse for the majority of content items, which likely explains why those features carried so little signal in the honest model (0.322 precision, barely above the 0.286 base rate for the minority class). I also can't tell from this single month whether a client's absence of GA4 data means "no traffic" or "GA4 wasn't connected for that client" — that distinction requires checking dim_clients, which this contract doesn't cover.

## Self-check
Before you submit, confirm each line honestly:
- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.